In [11]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/house-prices-advanced-regression-techniques/test.csv


In [12]:
df_train=pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
df_test=pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")
df_train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [13]:
# saving test Id before dropping it
# we'll need it later when creating the submission file
Id=df_test['Id']

In [14]:
df_train.describe().round()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
count,1460.0,1460.0,1201.0,1460.0,1460.0,1460.0,1460.0,1460.0,1452.0,1460.0,...,1460.0,1460.0,1460.0,1460.0,1460.0,1460.0,1460.0,1460.0,1460.0,1460.0
mean,730.0,57.0,70.0,10517.0,6.0,6.0,1971.0,1985.0,104.0,444.0,...,94.0,47.0,22.0,3.0,15.0,3.0,43.0,6.0,2008.0,180921.0
std,422.0,42.0,24.0,9981.0,1.0,1.0,30.0,21.0,181.0,456.0,...,125.0,66.0,61.0,29.0,56.0,40.0,496.0,3.0,1.0,79443.0
min,1.0,20.0,21.0,1300.0,1.0,1.0,1872.0,1950.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2006.0,34900.0
25%,366.0,20.0,59.0,7554.0,5.0,5.0,1954.0,1967.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,2007.0,129975.0
50%,730.0,50.0,69.0,9478.0,6.0,5.0,1973.0,1994.0,0.0,384.0,...,0.0,25.0,0.0,0.0,0.0,0.0,0.0,6.0,2008.0,163000.0
75%,1095.0,70.0,80.0,11602.0,7.0,6.0,2000.0,2004.0,166.0,712.0,...,168.0,68.0,0.0,0.0,0.0,0.0,0.0,8.0,2009.0,214000.0
max,1460.0,190.0,313.0,215245.0,10.0,9.0,2010.0,2010.0,1600.0,5644.0,...,857.0,547.0,552.0,508.0,480.0,738.0,15500.0,12.0,2010.0,755000.0


In [15]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [16]:
df_train.isna().sum()

Id                 0
MSSubClass         0
MSZoning           0
LotFrontage      259
LotArea            0
                ... 
MoSold             0
YrSold             0
SaleType           0
SaleCondition      0
SalePrice          0
Length: 81, dtype: int64

In [17]:
df_train=df_train.drop(columns=df_train.select_dtypes(include=['object']).columns)
#Drop object-type features
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 38 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   LotFrontage    1201 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   OverallQual    1460 non-null   int64  
 5   OverallCond    1460 non-null   int64  
 6   YearBuilt      1460 non-null   int64  
 7   YearRemodAdd   1460 non-null   int64  
 8   MasVnrArea     1452 non-null   float64
 9   BsmtFinSF1     1460 non-null   int64  
 10  BsmtFinSF2     1460 non-null   int64  
 11  BsmtUnfSF      1460 non-null   int64  
 12  TotalBsmtSF    1460 non-null   int64  
 13  1stFlrSF       1460 non-null   int64  
 14  2ndFlrSF       1460 non-null   int64  
 15  LowQualFinSF   1460 non-null   int64  
 16  GrLivArea      1460 non-null   int64  
 17  BsmtFullBath   1460 non-null   int64  
 18  BsmtHalf

In [18]:
df_columns_drops=['Id', 'MSSubClass', 'LotFrontage', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'LowQualFinSF', 'BsmtFullBath'
                  , 'BsmtHalfBath', 'KitchenAbvGr', 'GarageYrBlt', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'MiscVal', 'MoSold']
df_train=df_train.drop(columns=df_columns_drops)
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 21 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   LotArea       1460 non-null   int64
 1   OverallQual   1460 non-null   int64
 2   OverallCond   1460 non-null   int64
 3   YearBuilt     1460 non-null   int64
 4   YearRemodAdd  1460 non-null   int64
 5   TotalBsmtSF   1460 non-null   int64
 6   1stFlrSF      1460 non-null   int64
 7   2ndFlrSF      1460 non-null   int64
 8   GrLivArea     1460 non-null   int64
 9   FullBath      1460 non-null   int64
 10  HalfBath      1460 non-null   int64
 11  BedroomAbvGr  1460 non-null   int64
 12  TotRmsAbvGrd  1460 non-null   int64
 13  Fireplaces    1460 non-null   int64
 14  GarageCars    1460 non-null   int64
 15  GarageArea    1460 non-null   int64
 16  WoodDeckSF    1460 non-null   int64
 17  OpenPorchSF   1460 non-null   int64
 18  PoolArea      1460 non-null   int64
 19  YrSold        1460 non-null

In [19]:
df_train.head()

,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,TotalBsmtSF,1stFlrSF,2ndFlrSF,GrLivArea,FullBath,...,BedroomAbvGr,TotRmsAbvGrd,Fireplaces,GarageCars,GarageArea,WoodDeckSF,OpenPorchSF,PoolArea,YrSold,SalePrice
0,8450,7,5,2003,2003,856,856,854,1710,2,...,3,8,0,2,548,0,61,0,2008,208500
1,9600,6,8,1976,1976,1262,1262,0,1262,2,...,3,6,1,2,460,298,0,0,2007,181500
2,11250,7,5,2001,2002,920,920,866,1786,2,...,3,6,1,2,608,0,42,0,2008,223500
3,9550,7,5,1915,1970,756,961,756,1717,1,...,3,7,1,3,642,0,35,0,2006,140000
4,14260,8,5,2000,2000,1145,1145,1053,2198,2,...,4,9,1,3,836,192,84,0,2008,250000


In [20]:
df_train.isna().sum()#check the missing values

LotArea         0
OverallQual     0
OverallCond     0
YearBuilt       0
YearRemodAdd    0
TotalBsmtSF     0
1stFlrSF        0
2ndFlrSF        0
GrLivArea       0
FullBath        0
HalfBath        0
BedroomAbvGr    0
TotRmsAbvGrd    0
Fireplaces      0
GarageCars      0
GarageArea      0
WoodDeckSF      0
OpenPorchSF     0
PoolArea        0
YrSold          0
SalePrice       0
dtype: int64

In [21]:
X=df_train.drop(columns='SalePrice') #define features
y=df_train['SalePrice'] #define lables

In [22]:
from sklearn.model_selection import train_test_split #split df
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=44,
    shuffle=True
)


In [23]:
from sklearn.linear_model import LinearRegression #improt simple linear model
model=LinearRegression() #define model

In [24]:
model.fit(X_train,y_train) #model training 

LinearRegression()

In [25]:
test_predict=model.predict(X_test)#model predict the result of X_test

In [26]:
from sklearn.metrics import mean_squared_error #import RMSE 
RMSE=np.sqrt(mean_squared_error(test_predict,y_test)) 
print(RMSE) #30206 mean error price

30199.40201848837


In [27]:
# filling missing values in the test set using statistics from training data
# so the model sees data in the same format it was trained on
medians = df_train.median()
df_test = df_test.fillna(medians)

In [28]:
# new features
TotalHouseSF=df_train['TotalBsmtSF']+df_train['1stFlrSF']+df_train['2ndFlrSF']
# new feature total area 
Bath=df_train['HalfBath']+df_train['FullBath']
# new feature total bathroom
HouseAge=df_train['YrSold']-df_train['YearBuilt']
# new feature house age
RemodAge=df_train['YrSold']-df_train['YearRemodAdd']
#new feature remod age


In [29]:
df_train=df_train.drop(columns=['TotalBsmtSF','1stFlrSF','2ndFlrSF','HalfBath',
                       'FullBath', 'YrSold','YearBuilt','YrSold','YearRemodAdd'])

In [30]:
#add new features to df
df_train['TotalHouseSF']=TotalHouseSF
df_train['Bath']=Bath
df_train['HouseAge']=HouseAge
df_train['RemodAge']=RemodAge
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 17 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   LotArea       1460 non-null   int64
 1   OverallQual   1460 non-null   int64
 2   OverallCond   1460 non-null   int64
 3   GrLivArea     1460 non-null   int64
 4   BedroomAbvGr  1460 non-null   int64
 5   TotRmsAbvGrd  1460 non-null   int64
 6   Fireplaces    1460 non-null   int64
 7   GarageCars    1460 non-null   int64
 8   GarageArea    1460 non-null   int64
 9   WoodDeckSF    1460 non-null   int64
 10  OpenPorchSF   1460 non-null   int64
 11  PoolArea      1460 non-null   int64
 12  SalePrice     1460 non-null   int64
 13  TotalHouseSF  1460 non-null   int64
 14  Bath          1460 non-null   int64
 15  HouseAge      1460 non-null   int64
 16  RemodAge      1460 non-null   int64
dtypes: int64(17)
memory usage: 194.0 KB


In [31]:
df_train

,LotArea,OverallQual,OverallCond,GrLivArea,BedroomAbvGr,TotRmsAbvGrd,Fireplaces,GarageCars,GarageArea,WoodDeckSF,OpenPorchSF,PoolArea,SalePrice,TotalHouseSF,Bath,HouseAge,RemodAge
0,8450,7,5,1710,3,8,0,2,548,0,61,0,208500,2566,3,5,5
1,9600,6,8,1262,3,6,1,2,460,298,0,0,181500,2524,2,31,31
2,11250,7,5,1786,3,6,1,2,608,0,42,0,223500,2706,3,7,6
3,9550,7,5,1717,3,7,1,3,642,0,35,0,140000,2473,1,91,36
4,14260,8,5,2198,4,9,1,3,836,192,84,0,250000,3343,3,8,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,7917,6,5,1647,3,7,1,2,460,0,40,0,175000,2600,3,8,7
1456,13175,6,6,2073,3,7,2,2,500,349,0,0,210000,3615,2,32,22
1457,9042,7,9,2340,4,9,2,1,252,0,60,0,266500,3492,2,69,4
1458,9717,5,6,1078,2,5,0,1,240,366,0,0,142125,2156,1,60,14


In [32]:
X=df_train.drop(columns='SalePrice') #redifne x with new features
y=df_train['SalePrice'] #redifine y 
# split df
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=44,
    shuffle=True
)

In [33]:
model.fit(X_train,y_train)#Fiting

LinearRegression()

In [34]:
test_predict=model.predict(X_test)
RMSE=np.sqrt(mean_squared_error(test_predict,y_test))
print(RMSE)

30158.89696461497


In [35]:
X=df_train.drop(columns='SalePrice')
y=df_train['SalePrice']
X_train,X_test,y_train ,y_test= train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=44,
    shuffle=True
)
from sklearn.linear_model import Lasso
model=Lasso(alpha=1290)
model.fit(X_train,y_train)
# using Lasso mainly to inspect weak features
# not as the final predictive model

Lasso(alpha=1290)

In [36]:
test_predict=model.predict(X_test)
RMSE=np.sqrt(mean_squared_error(test_predict,y_test))
print(RMSE) 

29702.55592331735


In [37]:
# extracting feature weights from Lasso
# to identify which features contribute little to the model
lasso_coefs=model.coef_
coef_df=pd.DataFrame({
    'feature': X_train.columns,
    'coefficient':lasso_coefs
})
weak_features = coef_df[coef_df['coefficient'] == 0]
weak_features_names = weak_features['feature'].tolist()
print(weak_features_names)

['Bath']


In [38]:
df_train=df_train.drop(columns='Bath')
df_train

,LotArea,OverallQual,OverallCond,GrLivArea,BedroomAbvGr,TotRmsAbvGrd,Fireplaces,GarageCars,GarageArea,WoodDeckSF,OpenPorchSF,PoolArea,SalePrice,TotalHouseSF,HouseAge,RemodAge
0,8450,7,5,1710,3,8,0,2,548,0,61,0,208500,2566,5,5
1,9600,6,8,1262,3,6,1,2,460,298,0,0,181500,2524,31,31
2,11250,7,5,1786,3,6,1,2,608,0,42,0,223500,2706,7,6
3,9550,7,5,1717,3,7,1,3,642,0,35,0,140000,2473,91,36
4,14260,8,5,2198,4,9,1,3,836,192,84,0,250000,3343,8,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,7917,6,5,1647,3,7,1,2,460,0,40,0,175000,2600,8,7
1456,13175,6,6,2073,3,7,2,2,500,349,0,0,210000,3615,32,22
1457,9042,7,9,2340,4,9,2,1,252,0,60,0,266500,3492,69,4
1458,9717,5,6,1078,2,5,0,1,240,366,0,0,142125,2156,60,14


In [39]:
X=df_train.drop(columns='SalePrice')
y=df_train['SalePrice']
X_train,X_test,y_train ,y_test= train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=44,
    shuffle=True
)
# using Ridge to stabilize the linear model
from sklearn.linear_model import Ridge
model=Ridge(alpha=328)
model.fit(X_train,y_train)


Ridge(alpha=328)

In [40]:
predict=model.predict(X_test)
RMSE=np.sqrt(mean_squared_error(predict,y_test))
print(RMSE)


29428.892520925037


In [41]:
# switching to Gradient Boosting
# since house prices rarely follow a purely linear relationship
from sklearn.ensemble import GradientBoostingRegressor
model = GradientBoostingRegressor(
       n_estimators=178,
    learning_rate=0.1,
    max_depth=4,
    random_state=63
)
model.fit(X_train,y_train)

GradientBoostingRegressor(max_depth=4, n_estimators=178, random_state=63)

In [42]:
predict=model.predict(X_test)
RMSE=np.sqrt(mean_squared_error(predict,y_test))
print(RMSE)

26413.350666776256


In [43]:
df_test

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,2915,160,RM,21.0,1936,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2006,WD,Normal
1455,2916,160,RM,21.0,1894,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,4,2006,WD,Abnorml
1456,2917,20,RL,160.0,20000,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,9,2006,WD,Abnorml
1457,2918,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal


 applying the same preprocessing logic used during training
 to keep feature structure consistent

In [44]:
df_test.isna().sum()

Id                 0
MSSubClass         0
MSZoning           4
LotFrontage      227
LotArea            0
                ... 
MiscVal            0
MoSold             0
YrSold             0
SaleType           1
SaleCondition      0
Length: 80, dtype: int64

In [45]:
df_test=df_test.drop(columns=df_test.select_dtypes(include=['object']).columns)

df_test=df_test.drop(columns=df_columns_drops)

df_test.isna().sum()



LotArea         0
OverallQual     0
OverallCond     0
YearBuilt       0
YearRemodAdd    0
TotalBsmtSF     0
1stFlrSF        0
2ndFlrSF        0
GrLivArea       0
FullBath        0
HalfBath        0
BedroomAbvGr    0
TotRmsAbvGrd    0
Fireplaces      0
GarageCars      0
GarageArea      0
WoodDeckSF      0
OpenPorchSF     0
PoolArea        0
YrSold          0
dtype: int64

In [46]:
# new features
TotalHouseSF=df_test['TotalBsmtSF']+df_test['1stFlrSF']+df_test['2ndFlrSF']
# new feature total area 
HouseAge=df_test['YrSold']-df_test['YearBuilt']
# new feature house age
RemodAge=df_test['YrSold']-df_test['YearRemodAdd']
#new feature remod age

df_test=df_test.drop(columns=['TotalBsmtSF','1stFlrSF','2ndFlrSF','HalfBath',
                       'FullBath', 'YrSold','YearBuilt','YrSold','YearRemodAdd'])

df_test['TotalHouseSF']=TotalHouseSF
df_test['HouseAge']=HouseAge
df_test['RemodAge']=RemodAge
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   LotArea       1459 non-null   int64  
 1   OverallQual   1459 non-null   int64  
 2   OverallCond   1459 non-null   int64  
 3   GrLivArea     1459 non-null   int64  
 4   BedroomAbvGr  1459 non-null   int64  
 5   TotRmsAbvGrd  1459 non-null   int64  
 6   Fireplaces    1459 non-null   int64  
 7   GarageCars    1459 non-null   float64
 8   GarageArea    1459 non-null   float64
 9   WoodDeckSF    1459 non-null   int64  
 10  OpenPorchSF   1459 non-null   int64  
 11  PoolArea      1459 non-null   int64  
 12  TotalHouseSF  1459 non-null   float64
 13  HouseAge      1459 non-null   int64  
 14  RemodAge      1459 non-null   int64  
dtypes: float64(3), int64(12)
memory usage: 171.1 KB


In [47]:
# generating predictions for unseen test data
predicted=model.predict(df_test)
# formatting predictions according to Kaggle submission format
submission=pd.DataFrame({
    'id':Id,
    'SalePrice':predicted
})

In [48]:
submission.to_csv('submission.csv' ,index=False)
submission
# final public score: 0.14791 RMSLE

,id,SalePrice
0,1461,131839.525090
1,1462,168885.685491
2,1463,181725.953394
3,1464,184358.700677
4,1465,187453.166208
...,...,...
1454,2915,87391.321158
1455,2916,89610.892160
1456,2917,184342.330779
1457,2918,110567.714084
